# ME 481: Whole Body Biomechanics - Electromyography Lab

<!-- ---

## Table of Contents
1. [Introduction](#introduction)
2. [Lab Overview and Objectives](#objectives)
3. [Background](#background)
4. [Data Used in This Lab](#data)
5. [Library Imports and Setup](#imports)
6. [Download Data](#download)
7. [Load Data](#load)
8. [Plot Raw Data](#plot-raw)
9. [Remove Bias (DC Offset)](#bias)
10. [Rectify Signals](#rectify)
11. [Filtering](#filtering)
12. [Plot Filtered Data](#plot-filtered)
13. [Calculate MVC Threshold](#mvc-threshold)
14. [Find Activation Times](#activation)

---

<a id="introduction"></a> -->

## Introduction

Welcome to the Electromyography (EMG) lab for ME 481. In this lab, you will learn how to process and analyze EMG data collected from different muscles.

**How to use this notebook:**
- Read each section's background and instructions.
- Complete and run the code cells. **Note: You will be asked to complete some of the code yourself! These portions are denoted by #TODO**
- Answer the discussion questions.
- Submit this completed notebook (**as HTML and .ipynb**) to Canvas.

<a id="background"></a>

## Background

Electromyography (EMG) is a technique for measuring and analyzing the electrical activity produced by skeletal muscles. EMG provides insights into muscle activation patterns, coordination, and neuromuscular control during various movements and tasks.


EMG data is widely used in biomechanics for a variety of purposes, including:
- **Forward Dynamics Simulations:** Using EMG-derived muscle activation patterns as inputs to musculoskeletal models to predict movement.
- **Validation of Static Optimization:** Comparing experimentally measured muscle activations to those predicted by optimization algorithms that estimate muscle forces from motion data.
- **Fatigue and Performance Assessment:** Monitoring changes in EMG signals to assess muscle fatigue, recruitment strategies, and performance during repetitive or strenuous activities.
- **Rehabilitation and Assistive Device Design:** Informing the design and evaluation of prosthetics, orthotics, and rehabilitation protocols by understanding muscle function in healthy and impaired populations.


## Learning Objectives

By the end of this lab, you will be able to:
- Understand the basics of EMG data collection and analysis
- Preprocess and filter EMG signals
- Calculate muscle activation thresholds
- Visualize and interpret EMG data

<a id="data"></a>

## Data Used in This Lab

The data for this lab was collected for a research study on handcycling biomechanics. Surface EMG sensors were placed on the biceps, middle deltoid, upper trapezius, and supraspinatus muscles. 

There are **two** distint types of EMG data we will be examining in this lab. 
- Maximum voluntary contractions (MVCs) were recorded in various isometric (stationary) positions to measure specific muscle acticvation signals. There is one MVC data file for each muscle. 
- Exercise trial data (10 seconds of continuous handcycling at moderate intensity) from all the EMG sensors. There is one trial data file that contains EMG signals for four unique muscles.

<a id="imports"></a>

## 1. Library Imports and Setup

Import all necessary libraries for data analysis, visualization, and notebook interaction.

In [ ]:
# Standard library imports
import os  # For interacting with the operating system (e.g., file paths)
import io  # For handling streams and file-like objects
from io import StringIO  # For working with string buffers as file-like objects
import requests  # For making HTTP requests
import nbformat  # For working with Jupyter Notebook file formats
import subprocess  # For executing shell commands and processes
from nbconvert import HTMLExporter
from pathlib import Path  # For working with file system paths in an object-oriented way

# Third-party library imports
import numpy as np  # For numerical computations and array operations
import pandas as pd  # For data manipulation and analysis using DataFrames
from scipy.signal import welch, butter, filtfilt  # For signal processing (spectral density, filtering)
import matplotlib.pyplot as plt  # For creating plots and visualizations
import ipywidgets  # For creating interactive widgets in Jupyter notebooks
import ipywidgets as widgets  # IPython widgets with an alias for convenience
import IPython.display  # For rendering rich outputs (e.g., images, HTML)
from IPython.display import display, HTML  # For displaying outputs and rendering HTML in Jupyter notebooks

# Jupyter-specific magic commands
%matplotlib inline

# Set default plot parameters settings
plt.style.use('default')
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 'large'
plt.rcParams['ytick.labelsize'] = 'large'
plt.rcParams['font.family'] = 'Sans'
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['figure.figsize'] = 8, 6
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.autolayout'] = True
plt.rcParams['axes.grid'] = True
# Use the 'tab10' color palette for plots
tab_colors = plt.get_cmap('tab10').colors

<a id="download"></a>

## 2. Download Data

Download the required EMG data files or verify their presence in the data directory.

In [ ]:
def download_file(url, save_path):
    """
    Downloads a file from a given URL and saves it to a specified path.

    Args:
        url (str): The URL of the file to download.
        save_path (str): The local path where the file will be saved.

    Returns:
        str: The path where the file was saved.

    Raises:
        requests.exceptions.RequestException: If an error occurs during the download.
    """
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(save_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    return save_path


# Download URLs
biceps_mvc_url = 'https://uofi.box.com/shared/static/5nvtemlh3i6rwinpgohrbbyy0paanhnu.csv'
trap_mvc_url = 'https://uofi.box.com/shared/static/w7bbtxej4xvwh7h7659jogxtd5reefyo.csv'
deltoid_mvc_url = 'https://uofi.box.com/shared/static/ghqscvxxsiinmwdx68or01gfbcqghssz.csv'
supraspinatus_mvc_url = 'https://uofi.box.com/shared/static/bhj5ct8nqrlg7l1cf96zke6vuhy89la0.csv'
trial_data_url = 'https://uofi.box.com/shared/static/r2h7uxgm3svy5td1m295ibixganvv3f0.csv'

# Create 'data' directory if it doesn't exist
os.makedirs('data', exist_ok=True)
# Save paths (local)
biceps_mvc_data_path = 'data/Bicep_MVC.csv'
trap_mvc_data_path = 'data/Trap_MVC.csv'
deltoid_mvc_data_path = 'data/Delt_MVC.csv'
supraspinatus_mvc_data_path = 'data/Supra_MVC.csv'
trial_data_path = 'data/exercise_data.csv'

# Download files
# download_file(data_url, metabolic_data_path)

download_file(biceps_mvc_url, biceps_mvc_data_path)
download_file(trap_mvc_url, trap_mvc_data_path)
download_file(deltoid_mvc_url, deltoid_mvc_data_path)
download_file(supraspinatus_mvc_url, supraspinatus_mvc_data_path)
download_file(trial_data_url, trial_data_path)


if os.path.exists(trial_data_path) and os.path.exists(biceps_mvc_data_path) and os.path.exists(trap_mvc_data_path) and os.path.exists(deltoid_mvc_data_path) and os.path.exists(supraspinatus_mvc_data_path):
    print("Data files downloaded successfully!")
else:
    print("Failed to find downloaded data files.")

<a id="load"></a>

## 3. Load Data

Load the EMG data for each muscle and the trial task into Pandas DataFrames. We will filter the MVC data to only include the first 5 seconds.

In [ ]:
fs = 2000  # Sampling frequency in Hz

biceps_mvc_df = pd.read_csv(
    biceps_mvc_data_path, header=None, skiprows=1, usecols=[0, 1])
trap_mvc_df = pd.read_csv(
    trap_mvc_data_path, header=None, skiprows=1, usecols=[0, 1])
deltoid_mvc_df = pd.read_csv(
    deltoid_mvc_data_path, header=None, skiprows=1, usecols=[0, 1])
supraspinatus_mvc_df = pd.read_csv(
    supraspinatus_mvc_data_path, header=None, skiprows=1, usecols=[0, 1])
# Rename columns
biceps_mvc_df.columns = ['Time [s]', 'Biceps EMG [mV]']
trap_mvc_df.columns = ['Time [s]', 'Trap EMG [mV]']
deltoid_mvc_df.columns = ['Time [s]', 'Deltoid EMG [mV]']
supraspinatus_mvc_df.columns = ['Time [s]', 'Supraspinatus EMG [mV]']

# Only keep the first 5 seconds of MVC data for each muscle
biceps_mvc_df = biceps_mvc_df[biceps_mvc_df['Time [s]'] <= 5].reset_index(
    drop=True)
trap_mvc_df = trap_mvc_df[trap_mvc_df['Time [s]'] <= 5].reset_index(drop=True)
deltoid_mvc_df = deltoid_mvc_df[deltoid_mvc_df['Time [s]'] <= 5].reset_index(
    drop=True)
supraspinatus_mvc_df = supraspinatus_mvc_df[supraspinatus_mvc_df['Time [s]'] <= 5].reset_index(
    drop=True)


# Skip the first 3 rows of metadata
# Read only columns 9-12 (zero-based index 8-11) from the trial data, using row 3 as column names
trial_df = pd.read_csv(trial_data_path, header=4, usecols=range(8, 12))
trial_df.columns = ["Biceps EMG [mV]", "Trap EMG [mV]",
                    "Deltoid EMG [mV]", "Supraspinatus EMG [mV]"]
trial_df = trial_df.drop(index=0).reset_index(drop=True)
# Add a time column to the trial data based on the sampling rate (2000 Hz)
trial_df['Time [s]'] = trial_df.index / fs
# Move 'Time [s]' column to the front
cols = ['Time [s]'] + [col for col in trial_df.columns if col != 'Time [s]']
trial_df = trial_df[cols]

trial_df = trial_df[trial_df['Time [s]'] <= 10].reset_index(drop=True)
# Ensure all columns in the DataFrames are numeric (except 'Time [s]')
for df in [biceps_mvc_df, trap_mvc_df, deltoid_mvc_df, supraspinatus_mvc_df]:
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df.fillna(0, inplace=True)

for col in ["Biceps EMG [mV]", "Trap EMG [mV]", "Deltoid EMG [mV]", "Supraspinatus EMG [mV]"]:
    trial_df[col] = pd.to_numeric(trial_df[col], errors='coerce')
trial_df.fillna(0, inplace=True)

print("MVC Data:")
print(biceps_mvc_df.head())
print(trap_mvc_df.head())
print(deltoid_mvc_df.head())
print(supraspinatus_mvc_df.head())
print("\nTrial Data:")
print(trial_df.head())

<a id="plot-raw"></a>

## 4. Plot Raw Data

We will generate two sublplots to examine our raw MVC and trial data. 
- In the first subplot, we will visualize the unprocessed MVC EMG signals for each muscle. Note that even though these signals share the same time range, they were recorded separately.
- In the second subplot, we will visualize the unprocessed EMG signals for each muscle from a moderate intensity handcycling trial. These signals were all recorded at the same time.

*Why?* Plotting the raw data helps you identify noise, artifacts, and the general structure of the signals before any processing. This step provides a baseline for comparison after processing.

In [ ]:
# Start by plotting the MVC data for each muscle to visualize the raw EMG signals during the MVC trials
fig, axs = plt.subplots(4, 1, figsize=(8, 6), sharex=True)
axs[0].plot(biceps_mvc_df['Time [s]'],
            biceps_mvc_df['Biceps EMG [mV]'], label='Biceps MVC', color=tab_colors[0])
axs[0].set_title('Biceps MVC')
axs[0].set_ylabel('EMG (mV)')
axs[1].plot(trap_mvc_df['Time [s]'], trap_mvc_df['Trap EMG [mV]'],
            label='Trap MVC', color=tab_colors[1])
axs[1].set_title('Trap MVC')
axs[1].set_ylabel('EMG (mV)')
axs[2].plot(deltoid_mvc_df['Time [s]'],
            deltoid_mvc_df['Deltoid EMG [mV]'], label='Deltoid MVC', color=tab_colors[2])
axs[2].set_title('Deltoid MVC')
axs[2].set_ylabel('EMG (mV)')
axs[3].plot(supraspinatus_mvc_df['Time [s]'],
            supraspinatus_mvc_df['Supraspinatus EMG [mV]'], label='Supraspinatus MVC', color=tab_colors[3])
axs[3].set_title('Supraspinatus MVC')
axs[3].set_xlabel('Time (s)')
axs[3].set_ylabel('EMG (mV)')
plt.tight_layout()
plt.show()

In [ ]:
# Plot only seconds 1-10 of the trial data without using a mask variable
fig, axs = plt.subplots(4, 1, figsize=(8, 6), sharex=True)

axs[0].plot(
    trial_df.loc[(trial_df['Time [s]'] >= 1) & (
        trial_df['Time [s]'] <= 10), 'Time [s]'],
    trial_df.loc[(trial_df['Time [s]'] >= 1) & (
        trial_df['Time [s]'] <= 10), 'Biceps EMG [mV]'],
    color=tab_colors[0]
)
axs[0].set_title('Biceps EMG')
axs[0].set_ylabel('EMG (mV)')

axs[1].plot(
    trial_df.loc[(trial_df['Time [s]'] >= 1) & (
        trial_df['Time [s]'] <= 10), 'Time [s]'],
    trial_df.loc[(trial_df['Time [s]'] >= 1) & (
        trial_df['Time [s]'] <= 10), 'Trap EMG [mV]'],
    color=tab_colors[1]
)
axs[1].set_title('Trap EMG')
axs[1].set_ylabel('EMG (mV)')

axs[2].plot(
    trial_df.loc[(trial_df['Time [s]'] >= 1) & (
        trial_df['Time [s]'] <= 10), 'Time [s]'],
    trial_df.loc[(trial_df['Time [s]'] >= 1) & (
        trial_df['Time [s]'] <= 10), 'Deltoid EMG [mV]'],
    color=tab_colors[2]
)
axs[2].set_title('Deltoid EMG')
axs[2].set_ylabel('EMG (mV)')

axs[3].plot(
    trial_df.loc[(trial_df['Time [s]'] >= 1) & (
        trial_df['Time [s]'] <= 10), 'Time [s]'],
    trial_df.loc[(trial_df['Time [s]'] >= 1) & (
        trial_df['Time [s]'] <= 10), 'Supraspinatus EMG [mV]'],
    color=tab_colors[3]
)
axs[3].set_title('Supraspinatus EMG')
axs[3].set_xlabel('Time (s)')
axs[3].set_ylabel('EMG (mV)')

plt.tight_layout()
plt.show()

<a id="bias"></a>

## 5. Remove Bias (DC Offset)

Next we will subtract the mean value from each EMG signal to remove any DC offset (bias). 

*Why?* Removing the DC offset centers the signal around zero, which is important for accurate rectification and further analysis. Bias can be introduced by the electronics or electrode placement.

In [ ]:
# Remove bias (DC offset) by subtracting the mean from each EMG column in MVC and trial data

# Zero to mean for MVC data (only EMG columns, not "Time [s]")
biceps_mvc_df['Biceps EMG [mV]'] -= biceps_mvc_df['Biceps EMG [mV]'].mean()
trap_mvc_df['Trap EMG [mV]'] -= trap_mvc_df['Trap EMG [mV]'].mean()
deltoid_mvc_df['Deltoid EMG [mV]'] -= deltoid_mvc_df['Deltoid EMG [mV]'].mean()
supraspinatus_mvc_df['Supraspinatus EMG [mV]'] -= supraspinatus_mvc_df['Supraspinatus EMG [mV]'].mean()

# Zero to mean for trial data (all EMG columns except "Time [s]")
for col in ["Biceps EMG [mV]", "Trap EMG [mV]", "Deltoid EMG [mV]", "Supraspinatus EMG [mV]"]:
    trial_df[col] -= trial_df[col].mean()

<a id="filtering"></a>

## 6. Filtering

Next we will apply filters to remove noise while retaining frequencies associated with muscle activity.

*Why?* Filtering removes unwanted noise (like powerline interference) and isolates the frequency range relevant to muscle activity, improving the quality and interpretability of the EMG signals.

In [ ]:
# Remove 60 Hz powerline noise using a notch filter
def notch_filter(data, fs, freq=60.0, Q=30.0):
    """
    Apply a notch filter to remove powerline noise from EMG data.

    Args:
        data (array-like): The input EMG signal to be filtered.
        fs (float): The sampling frequency of the signal in Hz.
        freq (float): The frequency to be removed (default is 60 Hz).
        Q (float): The quality factor of the notch filter (default is 30).
    Returns:
        array-like: The filtered EMG signal with reduced powerline noise.
    """
    b, a = butter(4, [freq - freq / Q, freq + freq / Q],
                  btype='bandstop', fs=fs)
    filtered_data = filtfilt(b, a, data)
    return filtered_data

# Remove low-frequency movement artifacts and high-frequency noise using a bandpass filter


def bandpass_filter(data, fs, lowcut=20.0, highcut=450.0, order=4):
    """
    Apply a bandpass filter to EMG data to retain frequencies typically associated with muscle activity.

    Args:
        data (array-like): The input EMG signal to be filtered.
        fs (float): The sampling frequency of the signal in Hz.
        lowcut (float): The lower cutoff frequency of the bandpass filter (default is 20 Hz).
        highcut (float): The upper cutoff frequency of the bandpass filter (default is 450 Hz).
        order (int): The order of the Butterworth filter (default is 4).
    Returns:
        array-like: The filtered EMG signal with reduced noise and retained muscle activity frequencies.
    """
    b, a = butter(order, [lowcut, highcut], btype='band', fs=fs)
    filtered_data = filtfilt(b, a, data)
    return filtered_data


# Apply notch filter to MVC and trial data
biceps_mvc_df['Biceps EMG [mV]'] = notch_filter(
    biceps_mvc_df['Biceps EMG [mV]'], fs)
trap_mvc_df['Trap EMG [mV]'] = notch_filter(trap_mvc_df['Trap EMG [mV]'], fs)
deltoid_mvc_df['Deltoid EMG [mV]'] = notch_filter(
    deltoid_mvc_df['Deltoid EMG [mV]'], fs)
supraspinatus_mvc_df['Supraspinatus EMG [mV]'] = notch_filter(
    supraspinatus_mvc_df['Supraspinatus EMG [mV]'], fs)
for col in ["Biceps EMG [mV]", "Trap EMG [mV]", "Deltoid EMG [mV]", "Supraspinatus EMG [mV]"]:
    trial_df[col] = notch_filter(trial_df[col], fs)

# Apply bandpass filter to MVC and trial data
biceps_mvc_df['Biceps EMG [mV]'] = bandpass_filter(
    biceps_mvc_df['Biceps EMG [mV]'], fs)
trap_mvc_df['Trap EMG [mV]'] = bandpass_filter(
    trap_mvc_df['Trap EMG [mV]'], fs)
deltoid_mvc_df['Deltoid EMG [mV]'] = bandpass_filter(
    deltoid_mvc_df['Deltoid EMG [mV]'], fs)
supraspinatus_mvc_df['Supraspinatus EMG [mV]'] = bandpass_filter(
    supraspinatus_mvc_df['Supraspinatus EMG [mV]'], fs)
for col in ["Biceps EMG [mV]", "Trap EMG [mV]", "Deltoid EMG [mV]", "Supraspinatus EMG [mV]"]:
    trial_df[col] = bandpass_filter(trial_df[col], fs)

<a id="linear_envelope"></a>

## 7. Create the Linear Envelope

The "linear envelope" is a smoothed version of the rectified EMG signal, typically obtained by applying a low-pass filter (e.g., 10 Hz cutoff) to the rectified data. This process removes high-frequency fluctuations and reveals the overall amplitude profile of muscle activation over time. In this lab, we use a low-pass Butterworth filter to extract the linear envelope from the rectified EMG signals.

*Why?*
- It provides a clearer representation of the timing and intensity of muscle activation, making it easier to compare activation patterns between muscles or trials.
- The envelope is commonly used for detecting onset/offset of muscle activity, quantifying activation levels, and comparing EMG to other biomechanical signals (e.g., force, joint angle).



### 7a.) Rectification

First, convert all EMG signal values to their absolute values (full-wave rectification).

*Why?* Rectification makes all values positive, which is necessary because muscle activation is related to the magnitude of the signal, not its direction. This prepares the data for meaningful averaging and further processing.

In [ ]:
# Rectify (take absolute value) all EMG signals in MVC and trial data
biceps_mvc_df['Biceps EMG [mV]'] = biceps_mvc_df['Biceps EMG [mV]'].abs()
trap_mvc_df['Trap EMG [mV]'] = trap_mvc_df['Trap EMG [mV]'].abs()
deltoid_mvc_df['Deltoid EMG [mV]'] = deltoid_mvc_df['Deltoid EMG [mV]'].abs()
supraspinatus_mvc_df['Supraspinatus EMG [mV]'] = supraspinatus_mvc_df['Supraspinatus EMG [mV]'].abs()

for col in ["Biceps EMG [mV]", "Trap EMG [mV]", "Deltoid EMG [mV]", "Supraspinatus EMG [mV]"]:
    trial_df[col] = trial_df[col].abs()

### 7b.) Low-Pass Filter

Next, we apply a low-pass filter to the rectified EMG signal to obtain the linear envelope, which provides a smooth profile of muscle activation over time. 

*Why?*
- The rectified EMG signal contains rapid, high-frequency fluctuations due to the nature of muscle electrical activity.
- Low-pass filtering smooths out these rapid changes, leaving only the slower, overall trends in muscle activation.
- This makes it easier to interpret the timing and intensity of muscle activity, compare across muscles or trials, and relate EMG to other biomechanical signals (like force or joint angle).

In [ ]:
# Our signals have already been rectified, so we can apply a low-pass filter to extract the envelope of the EMG signals. We will use a cutoff frequency of around 10 Hz for the low-pass filter, which is commonly used for EMG envelope extraction.
def lowpass_filter(data, sampling_freq, cutoff_frequency=None):
    """
    Filter time series data using a low-pass Butterworth filter. This function applies a moving average smoothing filter before the Butterworth filter to each column of the input data besides the column named 'Time [s]', based on a specified or calculated cutoff frequency.

    Args:
        data (DataFrame): The input time series data to be filtered. Must be a pandas DataFrame.
        sampling_freq (float): The sampling frequency of the input data.
        custom_cutoff_frequency (float, optional): A user-defined cutoff frequency for the filter. If not provided, it will be determined using spectral analysis.
        threshold (float): Cumulative power threshold for automatic cutoff determination. Defaults to 99.
        plot (bool): Whether to plot spectral analysis results. Defaults to False.

    Returns:
        DataFrame: The filtered time series data, with the same structure as the input.

    Raises:
        ValueError: If the input data is not a pandas DataFrame.

    Examples:
        >>> filtered_data = filter_timeseries_data(data, sampling_freq=1000, threshold=95, plot=False)
    """
    if not isinstance(data, (pd.DataFrame)):
        raise ValueError(
            "Unsupported data type. Please provide a pandas DataFrame.")

    fs = float(sampling_freq)

    def apply_filter(column, cutoff_frequency):
        column = np.asarray(column)
        window_size = int(fs * 0.1)  # 100 ms window
        if window_size < 1:
            window_size = 1
        column = pd.Series(column).rolling(
            window=window_size, min_periods=1, center=True).mean().to_numpy()
        b, a = butter(N=4, Wn=cutoff_frequency,
                      btype="low", fs=fs, output="ba")
        return filtfilt(b, a, column)

    cutoff_frequencies = {
        column: cutoff_frequency for column in data.columns}

    # Only skip filtering for 'Time [s]' column
    filtered_data = data.apply(lambda column: apply_filter(
        column, cutoff_frequencies[column.name]) if column.name != "Time [s]" else column)

    return filtered_data


# Apply low-pass Butterworth filter to MVC data using the helper function
biceps_mvc_df_filtered = lowpass_filter(biceps_mvc_df, fs, cutoff_frequency=10)
trap_mvc_df_filtered = lowpass_filter(trap_mvc_df, fs, cutoff_frequency=10)
deltoid_mvc_df_filtered = lowpass_filter(
    deltoid_mvc_df, fs, cutoff_frequency=10)
supraspinatus_mvc_df_filtered = lowpass_filter(
    supraspinatus_mvc_df, fs, cutoff_frequency=10)

# Apply low-pass Butterworth filter to trial data using the helper function
trial_df_filtered = lowpass_filter(trial_df, fs, cutoff_frequency=10)

### 7c.) Plot the Linear Envelopes

Let's now plot the linear envelopes. You can refer back to section 4 to see how the signals have changed from the raw data.

[Quick link back to section 4](#plot-raw)

In [ ]:
# Plot the MVC linear envelope for each muscle
fig, axs = plt.subplots(1, 4, figsize=(8, 6), sharex=True)
axs[0].plot(biceps_mvc_df_filtered['Time [s]'],
            biceps_mvc_df_filtered['Biceps EMG [mV]'], label='Biceps MVC', color=tab_colors[0])
axs[0].set_title('Biceps MVC')
axs[0].set_ylabel('EMG (mV)')
axs[0].legend()
axs[1].plot(trap_mvc_df_filtered['Time [s]'], trap_mvc_df_filtered['Trap EMG [mV]'],
            label='Trap MVC', color=tab_colors[1])
axs[1].set_title('Trap MVC')
axs[1].set_ylabel('EMG (mV)')
axs[1].legend()
axs[2].plot(deltoid_mvc_df_filtered['Time [s]'],
            deltoid_mvc_df_filtered['Deltoid EMG [mV]'], label='Deltoid MVC', color=tab_colors[2])
axs[2].set_title('Deltoid MVC')
axs[2].set_ylabel('EMG (mV)')
axs[2].legend()
axs[3].plot(supraspinatus_mvc_df_filtered['Time [s]'],
            supraspinatus_mvc_df_filtered['Supraspinatus EMG [mV]'], label='Supraspinatus MVC', color=tab_colors[3])
axs[3].set_title('Supraspinatus MVC')
axs[3].set_xlabel('Time (s)')
axs[3].set_ylabel('EMG (mV)')
axs[3].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Plot the trial data linear envelope for each muscle
fig, axs = plt.subplots(4, 1, figsize=(8, 6), sharex=True)
axs[0].plot(
    trial_df_filtered['Time [s]'],
    trial_df_filtered['Biceps EMG [mV]'],
    color=tab_colors[0]
)
axs[0].set_title('Biceps EMG (Linear Envelope)')
axs[0].set_ylabel('EMG (mV)')
axs[1].plot(
    trial_df_filtered['Time [s]'],
    trial_df_filtered['Trap EMG [mV]'],
    color=tab_colors[1]
)
axs[1].set_title('Trap EMG (Linear Envelope)')
axs[1].set_ylabel('EMG (mV)')
axs[2].plot(
    trial_df_filtered['Time [s]'],
    trial_df_filtered['Deltoid EMG [mV]'],
    color=tab_colors[2]
)
axs[2].set_title('Deltoid EMG (Linear Envelope)')
axs[2].set_ylabel('EMG (mV)')
axs[3].plot(
    trial_df_filtered['Time [s]'],
    trial_df_filtered['Supraspinatus EMG [mV]'],
    color=tab_colors[3]
)
axs[3].set_title('Supraspinatus EMG (Linear Envelope)')
axs[3].set_xlabel('Time (s)')
axs[3].set_ylabel('EMG (mV)')
plt.tight_layout()
plt.show()

<a id="mvc-threshold"></a>

## 9. Calculate MVC Threshold

We will now shift gears to examine the MCV data. Our goal is to calculate a maximum voluntary contraction (MVC) value for each muscle. This is often paired with EMG data measured during some task of interest (in our case, handcycling). More on this later...

**Background: Why is MVC needed?**
- Raw EMG signals vary greatly between individuals, muscles, and even electrode placements, making direct comparison difficult.
- The MVC is the maximum electrical activity a muscle can produce during a voluntary contraction, serving as a personal reference point for each muscle and subject.
- By normalizing EMG data to the MVC (expressing activation as a percentage of MVC), we can:
  - Compare muscle activation levels across different muscles, people, and sessions.
  - Assess relative effort or muscle recruitment during tasks.
  - Make results more interpretable and meaningful in biomechanics and clinical contexts.

### 9a.) Define MVC Time Window

First, we define the start and end times for the window of interest in the MVC Trials. Use the plots above to visually identify a stable region of muscle activation during the MVC trials and use those time points to calculate the RMS values for each muscle's MVC data.

In [ ]:
# TODO: START_TIME =   # Start time of the window of interest in seconds
# TODO: END_TIME =     # End time of the window of interest in seconds

### 9b.) Calculate MVC Threshold

The root mean square (RMS) value is used to quantify the amplitude of the EMG signal during the maximum voluntary contraction (MVC) because EMG signals are highly variable and oscillate rapidly. The RMS calculation provides a single, consistent measure of the signal's overall intensity over a time window. By calculating the RMS during a stable portion of the MVC trial, we obtain a reliable reference value for normalizing EMG data.

*Why?*
- The RMS value reflects the effective magnitude of the EMG signal.
- It is less sensitive to random noise and outliers than simply taking the maximum value.
- Using RMS allows for a fair and robust comparison of muscle activation levels across different muscles, trials, and participants.

In [ ]:
def calculate_rms(signal_df, start_time=None, end_time=None):
    """
    Calculate the root mean square (RMS) of a 1D numpy array or pandas Series,
    optionally within a specified time window.

    Args:
        signal_df (pandas.DataFrame): DataFrame containing the EMG signal with a 'Time [s]' column and one or more EMG columns.
        start_time (float, optional): Start time in seconds for the window.
        end_time (float, optional): End time in seconds for the window.

    Returns:
        float: The RMS value over the specified window.
    """
    if start_time is not None and end_time is not None:
        mask = (signal_df['Time [s]'] >= start_time) & (
            signal_df['Time [s]'] <= end_time)
        signal_df = signal_df[mask]
    # Calculate RMS across all EMG columns
    return np.sqrt(np.mean(np.square(signal_df.iloc[:, 1:].values)))


biceps_rms = calculate_rms(
    biceps_mvc_df_filtered, start_time=START_TIME, end_time=END_TIME)
trap_rms = calculate_rms(
    trap_mvc_df_filtered, start_time=START_TIME, end_time=END_TIME)
deltoid_rms = calculate_rms(
    deltoid_mvc_df_filtered, start_time=START_TIME, end_time=END_TIME)
supraspinatus_rms = calculate_rms(
    supraspinatus_mvc_df_filtered, start_time=START_TIME, end_time=END_TIME)

print(f"Biceps MVC RMS: {biceps_rms:.6f} mV")
print(f"Trap MVC RMS: {trap_rms:.6f} mV")
print(f"Deltoid MVC RMS: {deltoid_rms:.6f} mV")
print(f"Supraspinatus MVC RMS: {supraspinatus_rms:.6f} mV")

### 9c.) Plot Calculated MVC Thresholds

In [ ]:
# Plot MVC data with RMS values annotated and RMS level as a dotted line
fig, axs = plt.subplots(4, 1, figsize=(8, 6), sharex=True)
axs[0].plot(biceps_mvc_df_filtered['Time [s]'],
            biceps_mvc_df_filtered['Biceps EMG [mV]'], label='Biceps MVC', color=tab_colors[0])
axs[0].axhline(biceps_rms, color='k', linestyle=':', label='RMS')
axs[0].set_title(f'Biceps MVC (RMS: {biceps_rms:.6f} mV)')
axs[0].set_ylabel('EMG (mV)')
axs[0].legend()
axs[1].plot(trap_mvc_df_filtered['Time [s]'], trap_mvc_df_filtered['Trap EMG [mV]'],
            label='Trap MVC', color=tab_colors[1])
axs[1].axhline(trap_rms, color='k', linestyle=':', label='RMS')
axs[1].set_title(f'Trap MVC (RMS: {trap_rms:.6f} mV)')
axs[1].set_ylabel('EMG (mV)')
axs[1].legend()
axs[2].plot(deltoid_mvc_df_filtered['Time [s]'],
            deltoid_mvc_df_filtered['Deltoid EMG [mV]'], label='Deltoid MVC', color=tab_colors[2])
axs[2].axhline(deltoid_rms, color='k', linestyle=':', label='RMS')
axs[2].set_title(f'Deltoid MVC (RMS: {deltoid_rms:.6f} mV)')
axs[2].set_ylabel('EMG (mV)')
axs[2].legend()
axs[3].plot(supraspinatus_mvc_df_filtered['Time [s]'],
            supraspinatus_mvc_df_filtered['Supraspinatus EMG [mV]'], label='Supraspinatus MVC', color=tab_colors[3])
axs[3].axhline(supraspinatus_rms, color='k',
               linestyle=':', label='RMS')
axs[3].set_title(f'Supraspinatus MVC (RMS: {supraspinatus_rms:.6f} mV)')
axs[3].set_xlabel('Time (s)')
axs[3].set_ylabel('EMG (mV)')
axs[3].legend()
plt.tight_layout()
plt.show()

<a id="activation"></a>

## 10. Find Activation Times

Next we will dentify when each muscle becomes active during the trial using a threshold based on the MVC.

*Why?* Determining activation times allows you to analyze muscle recruitment patterns and timing, which are important for understanding movement coordination and neuromuscular control.

### 10a.) Normalize exercise trial EMG data to %MVC

Next we will normalize the trial data to using calculated MVC values for each muscle. As a reminder, by normalizing to %MVC, you can compare muscle activation levels across different trials, muscles, or subjects, regardless of their absolute EMG signal strength. This makes the data more meaningful and comparable, as it accounts for individual differences in maximum muscle activation.

In [ ]:
# For each linear envelope trial signal, calculate the percentage of MVC activation by dividing the trial signal by the corresponding MVC RMS value and multiplying by 100
trial_df_normalized = trial_df_filtered.copy()
trial_df_normalized['Biceps EMG [% MVC]'] = (
    trial_df_filtered['Biceps EMG [mV]'] / biceps_rms) * 100
trial_df_normalized['Trap EMG [% MVC]'] = (
    trial_df_filtered['Trap EMG [mV]'] / trap_rms) * 100
trial_df_normalized['Deltoid EMG [% MVC]'] = (
    trial_df_filtered['Deltoid EMG [mV]'] / deltoid_rms) * 100
trial_df_normalized['Supraspinatus EMG [% MVC]'] = (
    trial_df_filtered['Supraspinatus EMG [mV]'] / supraspinatus_rms) * 100

### 10b.) Plot the MVC-Normalized Trial Data

In [ ]:
# Plot the normalized trial data as a percentage of MVC for each muscle
fig, axs = plt.subplots(4, 1, figsize=(8, 6), sharex=True)
axs[0].plot(
    trial_df_normalized['Time [s]'],
    trial_df_normalized['Biceps EMG [% MVC]'],
    color=tab_colors[0]
)
axs[0].set_title('Biceps EMG (% MVC)')
axs[0].set_ylabel('% MVC')
axs[1].plot(
    trial_df_normalized['Time [s]'],
    trial_df_normalized['Trap EMG [% MVC]'],
    color=tab_colors[1]
)
axs[1].set_title('Trap EMG (% MVC)')
axs[1].set_ylabel('% MVC')
axs[2].plot(
    trial_df_normalized['Time [s]'],
    trial_df_normalized['Deltoid EMG [% MVC]'],
    color=tab_colors[2]
)
axs[2].set_title('Deltoid EMG (% MVC)')
axs[2].set_ylabel('% MVC')
axs[3].plot(
    trial_df_normalized['Time [s]'],
    trial_df_normalized['Supraspinatus EMG [% MVC]'],
    color=tab_colors[3]
)
axs[3].set_title('Supraspinatus EMG (% MVC)')
axs[3].set_xlabel('Time (s)')
axs[3].set_ylabel('% MVC')
plt.tight_layout()
plt.show()

### 10c.) Identify Activation Times

We identify activation times by determining when the EMG signal crosses a specified threshold value because this method provides an objective and consistent way to detect when a muscle becomes active. The threshold is typically set based on baseline or noise levels, so when the signal exceeds this value, it indicates true muscle activation rather than random fluctuations or background noise. This approach helps ensure that the detected activation times are meaningful and reproducible.

**Based on the plot above**, choose an activation threshold (%MVC) that you feel minimizes false positive activations from noise but also doesn't miss any true activations.

In [ ]:
# TODO: ACTIVATION_THRESHOLD =  # Threshold for significant muscle activation as a percentage of MVC (0-100)

Find where the EMG signals exceed your chosen threshold activation level

In [ ]:
# Create boolean mask columns for each muscle indicating where the activation exceeds the threshold
trial_df_normalized['Biceps Active'] = trial_df_normalized['Biceps EMG [% MVC]'] >= ACTIVATION_THRESHOLD
trial_df_normalized['Trap Active'] = trial_df_normalized['Trap EMG [% MVC]'] >= ACTIVATION_THRESHOLD
trial_df_normalized['Deltoid Active'] = trial_df_normalized['Deltoid EMG [% MVC]'] >= ACTIVATION_THRESHOLD
trial_df_normalized['Supraspinatus Active'] = trial_df_normalized['Supraspinatus EMG [% MVC]'] >= ACTIVATION_THRESHOLD

Now, plot the normalized EMG data and highlight when the muscle is activated.

In [ ]:
# Plot the normalized trial data with activation masks
fig, axs = plt.subplots(4, 1, figsize=(8, 6), sharex=True)
axs[0].plot(trial_df_normalized['Time [s]'],
            trial_df_normalized['Biceps EMG [% MVC]'], label='Biceps EMG (% MVC)', color=tab_colors[0])
axs[0].fill_between(trial_df_normalized['Time [s]'], 0, trial_df_normalized['Biceps EMG [% MVC]'],
                    where=trial_df_normalized['Biceps Active'], color=tab_colors[0], alpha=0.3, label='Biceps Active')
axs[0].axhline(ACTIVATION_THRESHOLD, color='k',
               linestyle=':', label='Activation Threshold')
axs[0].set_title('Biceps EMG (% MVC)')
axs[0].set_ylabel('% MVC')
axs[0].legend(loc='upper right')
axs[1].plot(trial_df_normalized['Time [s]'],
            trial_df_normalized['Trap EMG [% MVC]'], label='Trap EMG (% MVC)', color=tab_colors[1])
axs[1].fill_between(trial_df_normalized['Time [s]'], 0, trial_df_normalized['Trap EMG [% MVC]'],
                    where=trial_df_normalized['Trap Active'], color=tab_colors[1], alpha=0.3, label='Trap Active')
axs[1].axhline(ACTIVATION_THRESHOLD, color='k',
               linestyle=':', label='Activation Threshold')
axs[1].set_title('Trap EMG (% MVC)')
axs[1].set_ylabel('% MVC')
axs[1].legend(loc='upper right')
axs[2].plot(trial_df_normalized['Time [s]'],
            trial_df_normalized['Deltoid EMG [% MVC]'], label='Deltoid EMG (% MVC)', color=tab_colors[2])
axs[2].fill_between(trial_df_normalized['Time [s]'], 0, trial_df_normalized['Deltoid EMG [% MVC]'],
                    where=trial_df_normalized['Deltoid Active'], color=tab_colors[2], alpha=0.3, label='Deltoid Active')
axs[2].axhline(ACTIVATION_THRESHOLD, color='k',
               linestyle=':', label='Activation Threshold')
axs[2].set_title('Deltoid EMG (% MVC)')
axs[2].set_ylabel('% MVC')
axs[2].legend(loc='upper right')
axs[3].plot(trial_df_normalized['Time [s]'],
            trial_df_normalized['Supraspinatus EMG [% MVC]'], label='Supraspinatus EMG (% MVC)', color=tab_colors[3])
axs[3].fill_between(trial_df_normalized['Time [s]'], 0, trial_df_normalized['Supraspinatus EMG [% MVC]'],
                    where=trial_df_normalized['Supraspinatus Active'], color=tab_colors[3], alpha=0.3, label='Supraspinatus Active')
axs[3].axhline(ACTIVATION_THRESHOLD, color='k',
               linestyle=':', label='Activation Threshold')
axs[3].set_title('Supraspinatus EMG (% MVC)')
axs[3].set_xlabel('Time (s)')
axs[3].set_ylabel('% MVC')
axs[3].legend(loc='upper right')
plt.tight_layout()
plt.show()

## Discussion Questions

1.) Based on the plot above, how many motion cycles (handcycle revolutions) do you think occured during this 10 second window?


2.) What do you notice about the pattern of activations across the different muscles?


3.) Is there one muscle that exhibits a different activation pattern? If so, which one and why do you think this might be?


4.) Are there any muscle activations in the plot above that exceed 100% MVC? What do you think this means?


5.) Based on the plot above, which muscles do you think contribute the most force during handcycling?


6.) Muscle activity in the shoulder is notoriously difficult to isolate because the muscles are very close together and even overlap, creating cross-talk? Which muscles in the data above d you think are most likely to exhibit signs of cross-talk?

YOUR ANSWERS GO HERE

1.)

2.)

3.)

4.)

5.)

6.)

## Export Notebook to HTML

In [ ]:
import nbformat
from nbconvert import HTMLExporter
from google.colab import files

# Helper function to export a Colab notebook as an HTML file
def export_current_notebook(notebook_path, output_filename="exported_notebook.html"):
    """
    Exports the specified Colab notebook (with all cell outputs) as an HTML file.
    """
    try:
        # Convert to HTML
        with open(notebook_path, "r", encoding="utf-8") as f:
            notebook_content = nbformat.read(f, as_version=4)

        # Convert notebook to HTML
        html_exporter = HTMLExporter()
        html_exporter.exclude_input = False
        html_exporter.exclude_output = False
        html_data, _ = html_exporter.from_notebook_node(notebook_content)

        # Add CSS to wrap lines and prevent clipping
        wrap_css = """
        <style>
            pre, code {
                word-wrap: break-word;
                white-space: pre-wrap;
                word-break: break-word;
            }
        </style>
        """
        html_data = wrap_css + html_data

        # Save the HTML file
        with open(output_filename, "w", encoding="utf-8") as f:
            f.write(html_data)

        print(f"Notebook exported successfully as {output_filename}")

        # Optionally download the file
        files.download(output_filename)

    except Exception as e:
        print(f"An error occurred: {e}")


# Change to your notebook path
notebook_to_export = #TODO: Copy the path to this notebook from the file tree and paste it here as a string
# This will save to your local downloads
output_filename = 'electromyography_lab.html' #TODO add your NetID to the filename to make it unique

export_current_notebook(notebook_path=notebook_to_export,
                        output_filename=output_filename)